# 5. Dataset Preparation

Builds train / validation / test splits from the labeled cluster inventory and validates label quality with a Random Forest baseline.

**Prerequisites:** `4.5. Enhanced Labeling Review.ipynb` completed with ≥ 200 labels per manual class.

**Steps:**
1. Load inventory, build label taxonomy from labeled data
2. Tile-based stratified train/val/test split (prevents same-tile leakage)
3. Random Forest baseline on hand-crafted features
4. Save `train.csv`, `val.csv`, `test.csv` for PointNet++ training

**Gate:** If RF macro-F1 < 70%, fix label quality before proceeding to notebook 6.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('.').resolve()))

from config import CLUSTERS_DIR, DATASET_DIR

DATASET_DIR.mkdir(parents=True, exist_ok=True)

MIN_CLASS_SAMPLES = 50    # classes below this are merged into NOISE
VAL_FRAC          = 0.15
TEST_FRAC         = 0.15
RANDOM_STATE      = 42

## 1. Load inventory and build label map

In [ ]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from utils.labels import Labels
from utils.cluster_dataset import ClusterDataset

inv = pd.read_csv(CLUSTERS_DIR / "inventory.csv")
labeled = inv[inv['final_label'].notna()].copy()
labeled['final_label'] = labeled['final_label'].astype(int)

# Drop Unknown and Noise — not meaningful training classes
labeled = labeled[~labeled['final_label'].isin([Labels.UNKNOWN, Labels.NOISE])].reset_index(drop=True)
print(f"Labeled clusters (excl. Unknown/Noise): {len(labeled)}")

# Load custom labels
custom_path = CLUSTERS_DIR / "custom_labels.json"
custom_labels = {}
if custom_path.exists():
    with open(custom_path) as f:
        custom_labels = {int(k): v for k, v in json.load(f).items()}

label_names = dict(Labels.STR_DICT)
label_names.update(custom_labels)

# Build label map (dynamic: include classes with >= MIN_CLASS_SAMPLES)
label_map = ClusterDataset.build_label_map(labeled, min_samples=MIN_CLASS_SAMPLES, merge_into=99)
print(f"\nLabel map ({len(set(label_map.values()))} classes):")
idx_to_code = {}
for code, idx in sorted(label_map.items()):
    idx_to_code[idx] = code
for idx in sorted(set(label_map.values())):
    code  = idx_to_code[idx]
    name  = label_names.get(code, str(code))
    count = (labeled['final_label'] == code).sum()
    print(f"  class {idx:>2d}  code {code:>3d}  {name:<22}  {count:>5} samples")

# Save label map for inference
with open(DATASET_DIR / "class_map.json", 'w') as f:
    json.dump({str(k): v for k, v in label_map.items()}, f)
print(f"\nSaved class_map.json")

## 2. Class distribution

In [ ]:
counts = labeled['final_label'].value_counts().sort_index()
names  = [label_names.get(int(c), str(c)) for c in counts.index]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(counts)), counts.values, color='steelblue')
ax.set_yticks(range(len(counts)))
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel('Number of labeled clusters')
ax.set_title('Label distribution')
plt.tight_layout()
plt.show()

## 3. Random stratified train / val / test split

Clusters are split randomly with stratification on the mapped label so every
class is proportionally represented in all three splits.

In [ ]:
from sklearn.model_selection import train_test_split

# Map labels first so stratification uses the merged classes
labeled['mapped_label'] = labeled['final_label'].map(label_map)

# Split off test set
train_val, test = train_test_split(
    labeled, test_size=TEST_FRAC, random_state=RANDOM_STATE,
    stratify=labeled['mapped_label']
)

# Split remaining into train + val
val_frac_adjusted = VAL_FRAC / (1 - TEST_FRAC)
train, val = train_test_split(
    train_val, test_size=val_frac_adjusted, random_state=RANDOM_STATE,
    stratify=train_val['mapped_label']
)

print(f"  Train:  {len(train)} clusters  ({train['tilecode'].nunique()} tiles)")
print(f"  Val:    {len(val)} clusters  ({val['tilecode'].nunique()} tiles)")
print(f"  Test:   {len(test)} clusters  ({test['tilecode'].nunique()} tiles)")

# Verify class coverage
train_classes = set(train['final_label'].unique())
test_classes  = set(test['final_label'].unique())
missing = train_classes - test_classes
if missing:
    missing_names = [label_names.get(int(c), str(c)) for c in missing]
    print(f"\n⚠ Classes missing from test set: {missing_names}")
    print("  Increase dataset size or lower MIN_CLASS_SAMPLES.")
else:
    print("\n✓ All classes appear in train, val, and test sets.")

In [ ]:
# Add split column and save
train = train.copy(); train['split'] = 'train'
val   = val.copy();   val['split']   = 'val'
test  = test.copy();  test['split']  = 'test'

for df, name in [(train, 'train'), (val, 'val'), (test, 'test')]:
    df.to_csv(DATASET_DIR / f"{name}.csv", index=False)
    print(f"Saved {name}.csv  ({len(df)} rows)")

## 4. Random Forest baseline

Trains on hand-crafted geometric + color features extracted from NPZ arrays.
**Gate:** macro-F1 < 70% suggests label issues or insufficient data.

In [ ]:
def extract_features(npz_path):
    """Extract hand-crafted features from a cluster NPZ."""
    try:
        data  = np.load(npz_path, allow_pickle=False)
        xyz   = data['xyz_centered'].astype(np.float32)
        rgb   = data['rgb_norm'].astype(np.float32)
        h_ag  = data['height_ag'].astype(np.float32)
    except Exception:
        return None

    x, y, z = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    dx = x.max() - x.min()
    dy = y.max() - y.min()
    width  = min(dx, dy)
    length = max(dx, dy)
    height = float(data['area_m2']) ** 0.5  # proxy; replace with real h if available
    height = h_ag.max() - h_ag.min() if len(h_ag) > 0 else 0.0

    return [
        float(dx),                  # bounding box X span
        float(dy),                  # bounding box Y span
        float(width),
        float(length),
        float(height),
        float(length / max(width, 0.01)),  # aspect ratio
        float(data.get('area_m2', np.array(0.0))),
        float(len(xyz)),            # n_points
        float(h_ag.mean()),
        float(h_ag.std()),
        float(h_ag.max()),
        float(rgb[:, 0].mean()),    # R mean
        float(rgb[:, 1].mean()),    # G mean
        float(rgb[:, 2].mean()),    # B mean
        float(rgb[:, 1].mean() - rgb[:, 0].mean()),  # greenness
    ]

FEATURE_NAMES = [
    'dx', 'dy', 'width', 'length', 'height', 'aspect', 'area',
    'n_pts', 'h_mean', 'h_std', 'h_max',
    'rgb_r', 'rgb_g', 'rgb_b', 'greenness',
]

print("Extracting features …")
feats_train, labels_train = [], []
feats_test,  labels_test  = [], []

for df, fl, ll in [(train, feats_train, labels_train), (test, feats_test, labels_test)]:
    for _, row in df.iterrows():
        f = extract_features(row['npz_path'])
        if f is None: continue
        fl.append(f)
        ll.append(label_map.get(int(row['final_label']), label_map.get(99, 0)))

X_train = np.array(feats_train, dtype=np.float32)
y_train = np.array(labels_train)
X_test  = np.array(feats_test,  dtype=np.float32)
y_test  = np.array(labels_test)

print(f"Train: {len(X_train)}  Test: {len(X_test)}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns
import joblib

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

class_names = [label_names.get(idx_to_code.get(i, 0), str(i))
               for i in range(len(set(label_map.values())))]

macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
print(f"Random Forest macro-F1: {macro_f1:.3f}")
if macro_f1 < 0.70:
    print("\n⚠  macro-F1 < 0.70 — label quality or data quantity issue.")
    print("   Fix labels in notebook 4.5 before proceeding to PointNet++.")
else:
    print("✓  Baseline passes the quality gate (F1 ≥ 0.70).")

print("\n" + classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

# Save RF model
from config import MODELS_DIR
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(rf, MODELS_DIR / "rf_baseline.pkl")
print(f"Saved RF model: {MODELS_DIR / 'rf_baseline.pkl'}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(max(6, len(class_names)), max(5, len(class_names))))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'RF Baseline — confusion matrix  (macro-F1={macro_f1:.2f})')
plt.tight_layout()
plt.savefig(DATASET_DIR / "rf_confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
# Feature importances
importances = pd.Series(rf.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(6, 5))
importances.plot.barh(ax=ax)
ax.set_title('RF feature importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()